# Text-Only BERT Bill Transfer Classifier

Train and evaluate a text-only BERT classifier for bill passage prediction using cross-country transfer:
- train on Canada, test on Canada and US
- train on US, test on US and Canada

Text inputs are built from title, description or long_description, and full bill text.

## 0. Setup

In [17]:
import json
import re
import subprocess
import zipfile
from dataclasses import dataclass
from pathlib import Path
from xml.etree import ElementTree as ET

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch import nn
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

ROOT = Path('..').resolve()
DATA = ROOT / 'data'
NORM = DATA / 'normalized'
CA_TXT = DATA / 'canada_bill_text'
US_TXT = DATA / 'us_bill_text'

MODEL_NAME = 'nlpaueb/legal-bert-base-uncased'
MAX_LENGTH = 256
CHUNK_STRIDE = 64
MAX_CHUNKS = 1
BATCH_SIZE = 8
EPOCHS = 10
LR = 2e-4
SEED = 42
TRAIN_FRAC = 0.7
VAL_FRAC = 0.1
TARGET_COL = 'passed'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'ROOT: {ROOT}')
print(f'MODEL_NAME: {MODEL_NAME}')
print(f'DEVICE: {DEVICE}')

ROOT: C:\Users\tamze\OneDrive\Documents\GitHub\cpsc440
MODEL_NAME: nlpaueb/legal-bert-base-uncased
DEVICE: cuda


## 1. Load Bills and Full Text

In [18]:
bills_path = NORM / 'bills.csv'
if not bills_path.exists():
    print('bills.csv not found - running normalize.py ...')
    result = subprocess.run(['python', str(ROOT / 'scripts' / 'normalize.py')], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError('normalize.py failed')

bills = pd.read_csv(bills_path, low_memory=False)
bills[TARGET_COL] = pd.to_numeric(bills[TARGET_COL], errors='coerce').fillna(0).astype(int)
bills['introduced_date'] = pd.to_datetime(bills.get('introduced_date'), errors='coerce')
bills['_row_id'] = np.arange(len(bills), dtype=int)

def _ca_xml_text(el) -> str:
    if el.tag in {'Identification', 'Label', 'MarginalNote'}:
        return ''
    if el.tag == 'AmendedText' and el.get('{http://www.w3.org/XML/1998/namespace}lang') == 'fr':
        return ''
    parts = []
    if el.tag == 'Text' and el.text:
        parts.append(el.text.strip())
    for child in el:
        parts.append(_ca_xml_text(child))
        if child.tail:
            parts.append(child.tail.strip())
    return ' '.join(p for p in parts if p)

def load_canada_texts(ca_txt_dir: Path) -> dict:
    texts = {}
    for session_dir in sorted(ca_txt_dir.iterdir()):
        if not session_dir.is_dir():
            continue
        session = session_dir.name
        for txt_file in session_dir.glob('*_english.txt'):
            bill_num = txt_file.name.replace('.pdf_english.txt', '').replace('_english.txt', '')
            texts[f'{session}/{bill_num}'] = txt_file.read_text(encoding='utf-8', errors='replace')
        for xml_file in session_dir.glob('*.xml'):
            key = f'{session}/{xml_file.stem}'
            if key in texts:
                continue
            try:
                root = ET.parse(xml_file).getroot()
                raw = _ca_xml_text(root)
                if raw:
                    texts[key] = raw
            except Exception as exc:
                print(f'Warning: {xml_file.name}: {exc}')
    return texts

_SKIP_TAGS = {'metadata', 'dublinCore', 'form', 'distribution-code', 'congress', 'session', 'current-chamber', 'action', 'action-date', 'action-desc', 'legis-type', 'enum', 'external-xref', 'quote'}
_PREFIX_TO_TYPE = {'HB': 'hr', 'SB': 's', 'HR': 'hres', 'SR': 'sres', 'HJR': 'hjres', 'SJR': 'sjres', 'HCR': 'hconres', 'SCR': 'sconres'}
_TYPE_TO_PREFIX = {v: k for k, v in _PREFIX_TO_TYPE.items()}

def _xml_text(el) -> str:
    if el.tag in _SKIP_TAGS:
        return ''
    parts = [el.text or '']
    for child in el:
        parts.append(_xml_text(child))
        parts.append(child.tail or '')
    return ' '.join(p.strip() for p in parts if p.strip())

def load_us_texts(us_txt_dir: Path) -> dict:
    texts = {}
    for zip_path in sorted(us_txt_dir.glob('*.zip')):
        parts = zip_path.stem.split('_')
        congress = parts[0]
        xml_type = parts[-1]
        legiscan_prefix = _TYPE_TO_PREFIX.get(xml_type)
        if legiscan_prefix is None:
            continue
        try:
            with zipfile.ZipFile(zip_path) as zf:
                for name in zf.namelist():
                    if not name.endswith('.xml'):
                        continue
                    match = re.match(rf'BILLS-{congress}{xml_type}(\d+)', name, re.IGNORECASE)
                    if not match:
                        continue
                    key = f'{congress}/{legiscan_prefix}{int(match.group(1))}'
                    try:
                        root = ET.fromstring(zf.read(name))
                        body = root.find('./legis-body')
                        if body is not None:
                            texts[key] = _xml_text(body)
                    except Exception:
                        pass
        except Exception as exc:
            print(f'Warning: {zip_path.name}: {exc}')
    return texts

ca_texts = load_canada_texts(CA_TXT)
us_texts = load_us_texts(US_TXT)

bills['full_text'] = ''
mask_ca = bills['source'] == 'canada'
mask_us = bills['source'] == 'us'
bills.loc[mask_ca, 'full_text'] = bills[mask_ca].apply(lambda r: ca_texts.get(f"{r['session']}/{r['bill_number']}", ''), axis=1)
bills.loc[mask_us, 'full_text'] = bills[mask_us].apply(lambda r: us_texts.get(f"{r['session']}/{r['bill_number']}", ''), axis=1)

def pick_first_present(df: pd.DataFrame, choices: list[str], default: str = '') -> pd.Series:
    out = pd.Series([default] * len(df), index=df.index, dtype='object')
    for col in choices:
        if col in df.columns:
            vals = df[col].fillna('').astype(str).str.strip()
            out = out.where(out.str.len() > 0, vals)
    return out

title_text = pick_first_present(bills, ['title'])
desc_text = pick_first_present(bills, ['long_description', 'description'])
full_text = bills['full_text'].fillna('').astype(str)
bills['model_text'] = (title_text + '\n' + desc_text + '\n' + full_text).str.replace(r'\s+', ' ', regex=True).str.strip()

print(f'Loaded {len(bills):,} bills')
print('Text coverage by source (model_text non-empty):')
print(bills.groupby('source')['model_text'].apply(lambda s: (s.str.len() > 0).mean()).round(3))

Loaded 120,339 bills
Text coverage by source (model_text non-empty):
source
canada    1.0
us        1.0
Name: model_text, dtype: float64


## 2. Country Splits

In [19]:
@dataclass
class CountrySplit:
    train_df: pd.DataFrame
    val_df: pd.DataFrame
    test_df: pd.DataFrame

def make_country_split(df: pd.DataFrame, source: str, train_frac: float = TRAIN_FRAC, val_frac: float = VAL_FRAC) -> CountrySplit:
    country_df = df[(df['source'] == source) & (df['model_text'].str.len() > 0)].copy()
    country_df = country_df.sort_values(['introduced_date', '_row_id']).reset_index(drop=True)
    if len(country_df) < 3:
        raise ValueError(f'Not enough rows for {source} split')

    n_total = len(country_df)
    n_train = max(1, int(n_total * train_frac))
    n_val = max(1, int(n_total * val_frac))
    if n_train + n_val >= n_total:
        n_train = max(1, n_total - 2)
        n_val = 1

    train_df = country_df.iloc[:n_train].copy()
    val_df = country_df.iloc[n_train:n_train + n_val].copy()
    test_df = country_df.iloc[n_train + n_val:].copy()
    if len(test_df) == 0:
        test_df = country_df.iloc[-1:].copy()

    return CountrySplit(train_df=train_df, val_df=val_df, test_df=test_df)

country_splits = {
    'canada': make_country_split(bills, 'canada'),
    'us': make_country_split(bills, 'us'),
}
for country, split in country_splits.items():
    print(f"{country.upper()}: train={len(split.train_df):,}, val={len(split.val_df):,}, test={len(split.test_df):,}")

CANADA: train=4,975, val=710, test=1,423
US: train=79,251, val=11,321, test=22,644


## 3. Tokenization and Dataset

In [20]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def chunk_text(text: str, tokenizer, max_length: int = MAX_LENGTH, stride: int = CHUNK_STRIDE):
    enc = tokenizer(
        text,
        add_special_tokens=True,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        padding='max_length',
        return_attention_mask=True,
        return_tensors='pt',
    )
    return enc['input_ids'], enc['attention_mask']

class TextOnlyBillDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)
        self.texts = self.df['model_text'].fillna('').astype(str).tolist()
        self.labels = self.df[TARGET_COL].to_numpy(dtype='float32')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return {
            'text': self.texts[idx],
            'label': torch.tensor(self.labels[idx], dtype=torch.float32),
        }

def collate_batch(batch):
    texts = [item['text'] for item in batch]
    labels = torch.stack([item['label'] for item in batch])

    chunked_ids = []
    chunked_masks = []
    max_chunks = 1
    for text in texts:
        input_ids, attention_mask = chunk_text(text, tokenizer)
        input_ids = input_ids[:MAX_CHUNKS]
        attention_mask = attention_mask[:MAX_CHUNKS]
        chunked_ids.append(input_ids)
        chunked_masks.append(attention_mask)
        max_chunks = max(max_chunks, input_ids.size(0))

    seq_len = chunked_ids[0].size(-1)
    padded_ids = []
    padded_masks = []
    for input_ids, attention_mask in zip(chunked_ids, chunked_masks):
        if input_ids.size(0) < max_chunks:
            pad_shape = (max_chunks - input_ids.size(0), seq_len)
            id_pad = torch.zeros(pad_shape, dtype=torch.long)
            mask_pad = torch.zeros(pad_shape, dtype=torch.long)
            input_ids = torch.cat([input_ids, id_pad], dim=0)
            attention_mask = torch.cat([attention_mask, mask_pad], dim=0)
        padded_ids.append(input_ids)
        padded_masks.append(attention_mask)

    return {
        'input_ids': torch.stack(padded_ids),
        'attention_mask': torch.stack(padded_masks),
        'labels': labels,
    }

## 4. Text-Only BERT Model

In [21]:
class TextOnlyBertClassifier(nn.Module):
    def __init__(self, model_name: str, bert_hidden_size: int = 768, mlp_hidden_dims: list[int] | None = None, dropout: float = 0.2, freeze_bert: bool = True):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        bert_hidden_size = getattr(self.bert.config, 'hidden_size', bert_hidden_size)
        if freeze_bert:
            for p in self.bert.parameters():
                p.requires_grad = False

        mlp_hidden_dims = mlp_hidden_dims or [256, 128]
        layers = []
        in_dim = bert_hidden_size
        for hidden_dim in mlp_hidden_dims:
            layers.extend([nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout)])
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, 1))
        self.classifier = nn.Sequential(*layers)

    def encode_text(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        batch_size, num_chunks, seq_len = input_ids.shape
        flat_ids = input_ids.view(batch_size * num_chunks, seq_len)
        flat_mask = attention_mask.view(batch_size * num_chunks, seq_len)
        outputs = self.bert(input_ids=flat_ids, attention_mask=flat_mask)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        cls_embeddings = cls_embeddings.view(batch_size, num_chunks, -1)
        return cls_embeddings.mean(dim=1)

    def forward(self, input_ids, attention_mask):
        text_vec = self.encode_text(input_ids, attention_mask)
        return self.classifier(text_vec).squeeze(-1)

print('TextOnlyBertClassifier defined.')

TextOnlyBertClassifier defined.


## 5. Training and Metrics

In [22]:
def make_pos_weight(labels: np.ndarray) -> torch.Tensor:
    positives = max(float(labels.sum()), 1.0)
    negatives = max(float(len(labels) - labels.sum()), 1.0)
    return torch.tensor([negatives / positives], dtype=torch.float32)

def make_loader(df: pd.DataFrame, shuffle: bool = False):
    ds = TextOnlyBillDataset(df)
    return torch.utils.data.DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=collate_batch,
        pin_memory=(DEVICE.type == 'cuda'),
    )

def train_one_epoch(model, loader, optimizer, criterion, device, grad_accum_steps=1, use_amp=False, scaler=None, show_progress=False, progress_desc='Train'):
    model.train()
    total_loss = 0.0
    total_examples = 0
    optimizer.zero_grad(set_to_none=True)
    iterator = tqdm(loader, desc=progress_desc, leave=False, dynamic_ncols=True) if show_progress else loader
    for step, batch in enumerate(iterator, start=1):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
            logits = model(input_ids, attention_mask)
            raw_loss = criterion(logits, labels)
            loss = raw_loss / grad_accum_steps

        if use_amp and scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if step % grad_accum_steps == 0 or step == len(loader):
            if use_amp and scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += raw_loss.item() * labels.size(0)
        total_examples += labels.size(0)
        if show_progress and hasattr(iterator, 'set_postfix'):
            iterator.set_postfix(loss=f'{raw_loss.item():.4f}')

    return total_loss / max(total_examples, 1)

def predict_loader(model, loader, device, use_amp=True):
    model.eval()
    all_logits = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            all_logits.append(logits.detach().cpu())
            all_labels.append(batch['labels'].detach().cpu())

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    probs = 1.0 / (1.0 + np.exp(-logits))
    return labels, probs, logits

def best_f1_threshold(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    if len(thresholds) == 0:
        return 0.5
    f1_vals = 2 * precision[:-1] * recall[:-1] / np.clip(precision[:-1] + recall[:-1], 1e-12, None)
    return float(thresholds[int(np.nanargmax(f1_vals))])

def compute_metrics(y_true, y_prob, threshold: float = 0.5) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'threshold': float(threshold),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'pr_auc': float(average_precision_score(y_true, y_prob)),
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
    }

## 6. Transfer Experiment Helpers

In [23]:
def train_country_text_model(train_df: pd.DataFrame, val_df: pd.DataFrame, train_source: str):
    train_loader = make_loader(train_df, shuffle=True)
    val_loader = make_loader(val_df, shuffle=False)

    model = TextOnlyBertClassifier(model_name=MODEL_NAME, freeze_bert=True).to(DEVICE)
    pos_weight = make_pos_weight(train_df[TARGET_COL].to_numpy(dtype='float32')).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LR)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

    history = []
    grad_accum_steps = max(1, 8 // max(1, BATCH_SIZE))
    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            DEVICE,
            grad_accum_steps=grad_accum_steps,
            use_amp=(DEVICE.type == 'cuda'),
            scaler=scaler,
            show_progress=True,
            progress_desc=f'{train_source.title()} train {epoch}/{EPOCHS}',
        )

        val_labels, val_probs, _ = predict_loader(model, val_loader, DEVICE)
        val_thr = best_f1_threshold(val_labels, val_probs)
        val_metrics = compute_metrics(val_labels, val_probs, threshold=val_thr)

        history.append({'epoch': epoch, 'train_loss': train_loss, **{k: v for k, v in val_metrics.items() if k != 'confusion_matrix'}})
        print(f"Epoch {epoch}/{EPOCHS} | train_loss={train_loss:.4f} | val_pr_auc={val_metrics['pr_auc']:.4f} | val_f1={val_metrics['f1']:.4f}")

    history_df = pd.DataFrame(history)
    final_val_labels, final_val_probs, _ = predict_loader(model, val_loader, DEVICE)
    threshold = best_f1_threshold(final_val_labels, final_val_probs)
    return model, threshold, history_df

def run_text_transfer_experiment(train_source: str, country_splits: dict[str, CountrySplit]) -> dict:
    train_split = country_splits[train_source]
    other_source = 'us' if train_source == 'canada' else 'canada'
    other_split = country_splits[other_source]

    model, threshold, history_df = train_country_text_model(train_split.train_df, train_split.val_df, train_source)

    in_loader = make_loader(train_split.test_df, shuffle=False)
    out_loader = make_loader(other_split.test_df, shuffle=False)

    in_y, in_prob, _ = predict_loader(model, in_loader, DEVICE)
    out_y, out_prob, _ = predict_loader(model, out_loader, DEVICE)

    in_metrics = compute_metrics(in_y, in_prob, threshold=threshold)
    out_metrics = compute_metrics(out_y, out_prob, threshold=threshold)

    rows = [
        {'train_country': train_source, 'test_country': train_source, 'setting': 'in_country_test', **in_metrics},
        {'train_country': train_source, 'test_country': other_source, 'setting': 'transfer_test', **out_metrics},
    ]

    return {
        'train_country': train_source,
        'other_country': other_source,
        'threshold': threshold,
        'history': history_df,
        'report': pd.DataFrame(rows),
    }

print('Text transfer helpers defined.')

Text transfer helpers defined.


## 7. Canada-Only Training Run

In [24]:
print('Running Canada-only text transfer experiment...')
canada_run = run_text_transfer_experiment('canada', country_splits)

print('\nCanada training history:')
display(canada_run['history'])

print('\nCanada transfer report:')
display(canada_run['report'])

Running Canada-only text transfer experiment...


Canada train 1/10:   0%|          | 0/622 [00:00<?, ?it/s]

Epoch 1/10 | train_loss=1.0077 | val_pr_auc=0.5078 | val_f1=0.4431


Canada train 2/10:   0%|          | 0/622 [00:00<?, ?it/s]

Epoch 2/10 | train_loss=0.8752 | val_pr_auc=0.5397 | val_f1=0.4748


...

Canada train 10/10:   0%|          | 0/622 [00:00<?, ?it/s]

Epoch 10/10 | train_loss=0.6755 | val_pr_auc=0.6186 | val_f1=0.5550

Canada training history:


,epoch,train_loss,threshold,accuracy,balanced_accuracy,pr_auc,roc_auc,f1,precision,recall
0,1,1.007705,0.647461,0.745070,0.638592,0.507757,0.675682,0.443077,0.444444,0.441718
1,2,0.875192,0.605469,0.750704,0.659476,0.539659,0.705723,0.474777,0.459770,0.490798
2,3,0.815617,0.600098,0.773239,0.678408,0.562856,0.723175,0.504615,0.506173,0.503067
3,4,0.774497,0.657227,0.784507,0.696487,0.575148,0.732719,0.532110,0.530488,0.533742
4,5,0.791817,0.594727,0.778873,0.690678,0.587821,0.742954,0.522796,0.518072,0.527607
5,6,0.735428,0.593750,0.774648,0.724543,0.620821,0.766725,0.562842,0.507389,0.631902
6,7,0.734659,0.601562,0.764789,0.726758,0.617103,0.765447,0.561680,0.490826,0.656442
7,8,0.714545,0.635254,0.787324,0.715543,0.619144,0.768279,0.557185,0.533708,0.582822
8,9,0.683159,0.753906,0.763380,0.687083,0.581339,0.734239,0.514451,0.486339,0.546012
9,10,0.675504,0.740234,0.749296,0.725317,0.618637,0.767931,0.555000,0.468354,0.680982



Canada transfer report:


,train_country,test_country,setting,threshold,accuracy,balanced_accuracy,pr_auc,roc_auc,f1,precision,recall,confusion_matrix
0,canada,canada,in_country_test,0.740234,0.864371,0.791416,0.495138,0.852602,0.303249,0.192661,0.711864,"[[1188, 176], [17, 42]]"
1,canada,us,transfer_test,0.740234,0.788332,0.537856,0.011612,0.502446,0.020838,0.010817,0.283333,"[[17800, 4664], [129, 51]]"


## 8. US-Only Training Run and Final Report

In [25]:
print('Running US-only text transfer experiment...')
us_run = run_text_transfer_experiment('us', country_splits)

print('\nUS training history:')
display(us_run['history'])

print('\nUS transfer report:')
display(us_run['report'])

final_report = pd.concat([canada_run['report'], us_run['report']], ignore_index=True)
final_report = final_report[[
    'train_country',
    'test_country',
    'setting',
    'threshold',
    'accuracy',
    'balanced_accuracy',
    'pr_auc',
    'roc_auc',
    'f1',
    'precision',
    'recall',
]]

print('\nCombined text-only transfer report:')
display(final_report)

report_path = NORM / 'text_only_bert_transfer_report.json'
with open(report_path, 'w', encoding='utf-8') as fp:
    json.dump(final_report.to_dict(orient='records'), fp, indent=2)
print(f'Saved transfer report to {report_path}')

Running US-only text transfer experiment...


Us train 1/10:   0%|          | 0/9907 [00:00<?, ?it/s]

Epoch 1/10 | train_loss=1.1928 | val_pr_auc=0.1709 | val_f1=0.2819


Us train 2/10:   0%|          | 0/9907 [00:00<?, ?it/s]

Epoch 2/10 | train_loss=1.0872 | val_pr_auc=0.1776 | val_f1=0.3051


...


US transfer report:


,train_country,test_country,setting,threshold,accuracy,balanced_accuracy,pr_auc,roc_auc,f1,precision,recall,confusion_matrix
0,us,us,in_country_test,0.973145,0.986751,0.591017,0.054233,0.694625,0.184783,0.180851,0.188889,"[[22310, 154], [146, 34]]"
1,us,canada,transfer_test,0.973145,0.958538,0.500000,0.102328,0.754219,0.000000,0.000000,0.000000,"[[1364, 0], [59, 0]]"



Combined text-only transfer report:


,train_country,test_country,setting,threshold,accuracy,balanced_accuracy,pr_auc,roc_auc,f1,precision,recall
0,canada,canada,in_country_test,0.740234,0.864371,0.791416,0.495138,0.852602,0.303249,0.192661,0.711864
1,canada,us,transfer_test,0.740234,0.788332,0.537856,0.011612,0.502446,0.020838,0.010817,0.283333
2,us,us,in_country_test,0.973145,0.986751,0.591017,0.054233,0.694625,0.184783,0.180851,0.188889
3,us,canada,transfer_test,0.973145,0.958538,0.500000,0.102328,0.754219,0.000000,0.000000,0.000000


Saved transfer report to C:\Users\tamze\OneDrive\Documents\GitHub\cpsc440\data\normalized\text_only_bert_transfer_report.json
